# Zero-shot transfer

> Score the OSSL models on every ring-trial instrument, without adapting them

In [ ]:
#| default_exp zero_shot

In [ ]:
#| export
import numpy as np, pandas as pd
from soilspecdl.all import predict, regression_scores
from ossl_rt_transfer.pretrain import rt_wns, log_props, preprocess

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
from fastcore.test import *
from soilspecdata.datasets.ringtrial import load_ringtrial
from ossl_rt_transfer.data import rt_path, load_instruments, ossl_props
from ossl_rt_transfer.pretrain import load_pretrained

The cells that read the ring-trial data are marked `#| rtdata`. The outputs on this page come from a local run.

## Scoring

In [ ]:
#| export
def zero_shot(
    model, # Pretrained model, such as from `load_pretrained`
    spectra, # `Spectra` of one ring-trial instrument, on `rt_wns`
    prop, # Ring-trial property, a key of `ossl_props`
)->dict: # Number of soils, and `regression_scores`
    "Score `model` on the soils of one instrument that have a value of `prop`, with no adaptation to the instrument"
    X, y, _ = spectra.xy(prop)
    y = np.log1p(y.ravel()) if prop in log_props else y.ravel()
    p = predict(model, preprocess().fit_transform(X), rt_wns)
    return {'n': len(y), **regression_scores(y, p, inv=np.expm1 if prop in log_props else None)}

`zero_shot` preprocesses the spectra as in pretraining, and scores the predictions against the measured values. For K, R², `r` and CCC are on `log1p(K)`, as in pretraining, and RMSE and bias are in cmolc/kg. The 38 ring-trial soils that are in OSSL were left out of pretraining, so every soil is a test soil.

In [ ]:
#| rtdata
rtd = load_ringtrial(rt_path())
models = {p: load_pretrained(f'ossl_{p}_snv_ep80') for p in ossl_props}
zs = pd.DataFrame([{'instrument': k, 'property': p, **zero_shot(models[p], s, p)} for k, s in rtd.items() for p in ossl_props])
test_eq(len(zs), 21 * 3)
clusters = load_instruments().cluster.reindex(list(rtd)).astype('Int64')
by_inst = lambda col: zs.pivot(index='instrument', columns='property', values=col).assign(cluster=clusters).sort_values(['cluster', 'pH_H20'])

## Results

R² by instrument, grouped by the spectral clusters of Safanelli et al. (2023). Hokudai is not in the instrument table, so it has no cluster:

In [ ]:
#| rtdata
by_inst('R²').round(2)

property,clay_perc,pH_H20,potassium_cmolkg,cluster
instrument,,,,
UWisc_fine,0.47,0.81,0.03,1
Lesotho,-0.24,-0.16,-1.46,2
UIUC,-0.01,0.38,-2.74,2
IAEA_aug2022,-2.73,0.53,-2.57,2
OSU,0.49,0.83,-0.73,2
UGhent,0.35,0.84,-1.39,2
Argonne,0.72,0.86,0.23,2
Woodwell_vertex,0.72,0.72,-0.24,3
ETH_alpha_1,0.32,0.76,-0.50,3


Correlation `r`:

In [ ]:
#| rtdata
by_inst('r').round(2)

property,clay_perc,pH_H20,potassium_cmolkg,cluster
instrument,,,,
UWisc_fine,0.78,0.91,0.60,1
Lesotho,0.32,0.38,-0.04,2
UIUC,0.51,0.81,0.36,2
IAEA_aug2022,0.53,0.88,0.69,2
OSU,0.72,0.92,0.66,2
UGhent,0.81,0.93,0.71,2
Argonne,0.85,0.94,0.72,2
Woodwell_vertex,0.85,0.89,0.73,3
ETH_alpha_1,0.71,0.89,0.75,3


Bias, as mean prediction minus mean measured value: clay in percentage points, pH in pH units, K in cmolc/kg:

In [ ]:
#| rtdata
by_inst('bias').round(2)

property,clay_perc,pH_H20,potassium_cmolkg,cluster
instrument,,,,
UWisc_fine,3.66,-0.17,0.01,1
IAEA_aug2022,-19.05,-0.56,0.77,2
UIUC,0.08,-0.56,0.63,2
Argonne,-0.02,-0.14,0.14,2
OSU,0.76,-0.07,0.41,2
UGhent,4.24,-0.03,0.53,2
Lesotho,-3.76,0.21,0.14,2
Woodwell_vertex,0.19,-0.27,0.32,3
Woodwell_alpha,0.63,-0.23,0.29,3


The median over the 21 instruments, and the KSSL instrument, whose spectra are like those of OSSL:

In [ ]:
#| rtdata
cols = ['R²', 'r', 'RMSE', 'bias']
pd.concat({'median': zs.groupby('property')[cols].median(), 'KSSL': zs[zs.instrument == 'KSSL'].set_index('property')[cols]}, axis=1).round(2)

median                    KSSL                  
                     R²     r  RMSE  bias    R²     r  RMSE  bias
property                                                         
clay_perc          0.57  0.83  7.67  0.76  0.76  0.93  5.76  3.30
pH_H20             0.81  0.92  0.53 -0.15  0.82  0.92  0.51 -0.14
potassium_cmolkg  -0.24  0.72  0.55  0.28  0.21  0.79  0.47  0.12